# SAC Irrigation Training - v2.10.0 E2 (Colab Pro)

**Algorithm:** TQC (sb3_contrib) with VDN-per-quantile critic  
**Architecture:** v2.7 observation layout (1097-dim, 8 features/agent)  
**Change vs v2.7:** scalar twin-Q -> 25-quantile twin critic with `top_quantiles_to_drop_per_net=5`; otherwise identical hyperparameters (ent_coef=0.05 fixed, LR 3e-4 -> 5e-5, twin critic).

## What E2 tests
Hypothesis: the deadly-triad cascade in v2.7 (Chapter 4 Section 4.4.7) is driven by optimistic-tail amplification of the Q-target.  Truncating the top 20% of the predicted quantiles at each gradient step breaks the positive-feedback loop structurally.

## Acceptance (v2.10 handoff Section 6)
- bias_ratio at step 250k within +/- 10% of 1.0  (0.90 <= Q_pred/R_real <= 1.10)
- dry / moderate yields within 1% of v2.7's published numbers
- spatial action std stays > 0.1 mm throughout training

In [ ]:
# Cell 1: Mount Google Drive, clone repo, install deps.
import subprocess, sys, os

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = '/content/drive/MyDrive/thesis_results'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive mounted. Results -> {DRIVE_ROOT}')

if os.path.exists('/content/thesis'):
    subprocess.run(['rm', '-rf', '/content/thesis'], check=True)
subprocess.run(
    ['git', 'clone', 'https://github.com/taratorbati/thesis.git', '/content/thesis'],
    check=True
)

# Switch to v2.10 branch if available
branch_check = subprocess.run(
    'cd /content/thesis && git checkout v2.10 2>/dev/null',
    shell=True, capture_output=True, text=True
)
if branch_check.returncode != 0:
    print('NOTE: v2.10 branch not found on remote yet. Using main.')
    print('      Commit and push the v2.10 files before running this notebook for real.')

os.chdir('/content/thesis')
sys.path.insert(0, '/content/thesis')

# Install matching versions of stable-baselines3 and sb3-contrib.
# NOTE: if sb3-contrib==2.6.0 is not on PyPI yet, try 2.5.0 or 2.4.0.
subprocess.run(
    ['pip', 'install', '--quiet',
     'stable-baselines3==2.6.0', 'sb3-contrib==2.6.0',
     'gymnasium', 'wandb', 'pytest'],
    check=True
)

import torch
print(f'PyTorch:           {torch.__version__}')
print(f'CUDA available:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:               {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2: WandB secret + GPU check.
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('OK  WANDB_API_KEY loaded from Colab Secrets.')
except Exception as e:
    print(f'NOTE: Could not load WANDB_API_KEY ({type(e).__name__}).')
    print('     Training continues without WandB - add the key to Colab Secrets to enable it.')

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi failed - no GPU allocated')

In [ ]:
# Cell 3: Pre-training validation (MUST PASS before Cell 4).
#
# Runs smoke tests, factorized-critic tests (v2.7), TQC-critic tests (v2.10),
# and a 1000-step pilot to detect import/wiring bugs.
# Abort if anything fails.

import subprocess, sys

print('Smoke tests...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'SMOKE TESTS FAILED'

print('\nFactorized-critic tests (v2.7 baseline)...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_factorized_critic.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'V2.7 FACTORIZED CRITIC TESTS FAILED'

print('\nTQC critic tests (v2.10)...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_tqc_critic.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'V2.10 TQC CRITIC TESTS FAILED'

print('\n1000-step pilot training (wiring check, ~1 minute)...')
from src.rl.train_v210_e2 import train_tqc_e2
_ = train_tqc_e2(
    seed=999,
    output_dir='/content/pilot',
    wandb_project=None,
    total_timesteps=1000,
)
print('\nOK  Pre-flight passed. Proceed to Cell 4.')

In [ ]:
# Cell 4: Full 250k training (TQC v2.10 E2, ~30-55 min on A100, ~2.5 h on T4).
#
# Start with SEED=0 (paired with v2.7 seed 0 published numbers).
# Expand seeds only after seed-0 results are evaluated.

SEED = 0   # CHANGE per session

from src.rl.train_v210_e2 import train_tqc_e2

model = train_tqc_e2(
    seed=SEED,
    output_dir='/content/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    enable_per_cell_eval=False,
)
print('Training complete.')

In [ ]:
# Cell 5: Copy results to Google Drive (excluding replay buffer).
import shutil, os, datetime

src = f'/content/thesis/results/rl/sac_v210_e2_seed{SEED}'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = f'{DRIVE_ROOT}/sac_v210_e2_seed{SEED}_{timestamp}'

shutil.copytree(src, dst, ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print(f'Saved to: {dst}')
print()
for root, _, files in os.walk(dst):
    for f in files:
        p = os.path.join(root, f)
        size = os.path.getsize(p)
        rel  = os.path.relpath(p, dst)
        print(f'  {rel}  ({size/1024:.1f} KB)')

In [ ]:
# Cell 6: Post-training 9-cell evaluation.
import subprocess, sys

model_path = f'/content/thesis/results/rl/sac_v210_e2_seed{SEED}/best_model/best_model.zip'

print('Evaluating on 9-cell grid (perfect forecast)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl_tqc',
    '--model', model_path,
    '--scenario', 'all',
    '--budget',   'all',
    '--forecast', 'perfect',
], capture_output=False)
assert r.returncode == 0, 'PERFECT-FORECAST EVAL FAILED'

print('\nEvaluating on 9-cell grid (noisy forecast, seed=42)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl_tqc',
    '--model', model_path,
    '--scenario', 'all',
    '--budget',   'all',
    '--forecast', 'noisy',
    '--noise-seed', '42',
], capture_output=False)
if r.returncode != 0:
    print('Noisy-forecast eval failed; perfect-forecast only.')

In [ ]:
# Cell 7: Bias-ratio trajectory plot.
import pandas as pd
import matplotlib.pyplot as plt

csv_path = f'/content/thesis/results/rl/sac_v210_e2_seed{SEED}/bias_ratio_log.csv'
df = pd.read_csv(csv_path)
print(df.tail(10))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df['step'], df['bias_ratio_mean'], '-o', label='bias_ratio_mean')
ax.axhline(1.0,  color='k', linestyle=':', alpha=0.5, label='ideal')
ax.axhline(1.10, color='r', linestyle=':', alpha=0.5, label='+/-10% threshold')
ax.axhline(0.90, color='r', linestyle=':', alpha=0.5)
ax.set_xlabel('training step')
ax.set_ylabel('bias ratio = Q_pred / R_realised')
ax.set_title(f'v2.10 E2 seed {SEED} - cascade diagnostic')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/bias_ratio_seed{SEED}.png', dpi=120)
plt.show()

In [ ]:
# Cell 8: Resume from Drive checkpoint (if session was interrupted).
# Uncomment and fill in CHECKPOINT_STEP / CHECKPOINT_DRIVE_PATH.

# SEED = 0
# CHECKPOINT_STEP = 100_000
# CHECKPOINT_DRIVE_PATH = f'{DRIVE_ROOT}/sac_v210_e2_seed{SEED}_YYYYMMDD_HHMMSS'
#
# import shutil, os
# local_dir = f'/content/thesis/results/rl/sac_v210_e2_seed{SEED}'
# os.makedirs(local_dir, exist_ok=True)
# shutil.copytree(CHECKPOINT_DRIVE_PATH, local_dir, dirs_exist_ok=True)
#
# from sb3_contrib import TQC
# from src.rl.networks_tqc import V27TQCPolicy
# ckpt = f'{local_dir}/checkpoints/sac_v210_e2_seed{SEED}_{CHECKPOINT_STEP}_steps.zip'
# model = TQC.load(ckpt, custom_objects={'policy_class': V27TQCPolicy})
# # Continue training with model.learn(total_timesteps=..., reset_num_timesteps=False)